In [ ]:
import os
import sys
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
from model.hopfield_network import HopfieldNetwork
from notebooks.plot_patterns import plot_patterns

In [ ]:
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))

if project_root not in sys.path:
    sys.path.append(project_root)

In [ ]:
data_name = "small-7x7"

patterns_path = Path(project_root) / f"data/{data_name}.csv"
plot_dim = (7,7)

df = pd.read_csv(patterns_path, header=None)
patterns = df.to_numpy(dtype=int)
n_patterns, n = patterns.shape
plot_patterns(patterns, plot_dim)

In [ ]:
train_seeds = [42, 43, 44, 45, 46]

noise_levels = np.arange(0, 0.31, 0.02)
noise_per_pattern = 20
max_iter = 100

recall_seed = 456
noise_seed = 123

methods = ["hebb","oji"]
modes = ["sync", "async"]
if_asynchronic = [True, False]
epochs = 30
lrs = np.arange(0.001, 0.011, 0.001).tolist()

output_dir = Path(f"{data_name}-results")
output_dir.mkdir(parents=True, exist_ok=True)

## Wyznaczanie optymalnego współczynnika uczenia

In [ ]:
lr_results = []

for lr in lrs:
    for seed in train_seeds:

        hn = HopfieldNetwork(n=n, method="oji", if_asynchronic=True, lr=lr, epochs=epochs, random_state=seed)
        hn.train(patterns)

        stable_count = sum(hn.check_if_stable(pattern) for pattern in patterns)

        lr_results.append({
            "Learning rate": float(lr),
            "Train seed": seed,
            "Stable patterns": stable_count,
            "Stable (%)": 100 * stable_count / n_patterns
        })

df_lr = pd.DataFrame(lr_results)

display(df_lr)


In [ ]:
df_lr_summary = (df_lr.groupby("Learning rate", as_index=False).agg(
    Mean_stable=("Stable patterns", "mean"),
    Std_stable=("Stable patterns", "std"),
    Mean_stable_pct=("Stable (%)", "mean")
))

best = df_lr_summary.loc[df_lr_summary["Mean_stable"].idxmax()]
best_lr = best["Learning rate"]

print(f"Best learning rate: {best_lr:.4f}")
print(
    f"Mean stable patterns: "
    f"{best['Mean_stable']:.2f}/{n_patterns}"
)

display(df_lr_summary)

## Trenowanie i sprawdzanie stabilności wzorców

In [ ]:
models = {}

for method in methods:
    seeds = train_seeds if method == "oji" else [42]

    for seed in seeds:
        hn = HopfieldNetwork(n=n, method=method, lr=lr, if_asynchronic=True, epochs=epochs, random_state=seed) 
        hn.train(patterns)

        models[(method, seed)] = hn

In [ ]:
stab_results = []

for (method, seed), hn in models.items():
    for i in range(n_patterns):
        stable = hn.check_if_stable(patterns[i])

        stab_results.append({
            "Method": method,
            "Train seed": seed,
            "Pattern": i,
            "Stable": stable
        })

df_stab = pd.DataFrame(stab_results)

display(df_stab)

In [ ]:
df_stab_per_seed = (df_stab.groupby(["Method", "Train seed"], as_index=False).agg(
    Stable_patterns=("Stable", "sum"),
    Total_patterns=("Stable", "count")
))

df_stab_per_seed["Stable (%)"] = (
    df_stab_per_seed["Stable_patterns"] / df_stab_per_seed["Total_patterns"] * 100
)

df_stab_summary = (
    df_stab_per_seed.groupby("Method", as_index=False).agg(
        Mean_stable=("Stable_patterns", "mean"),
        Std_stable=("Stable_patterns", "std"),
        Mean_stable_pct=("Stable (%)", "mean")
    )
)

display(df_stab_per_seed)
display(df_stab_summary)

## Przykładowe wizualizacje

In [ ]:
hn_hebb = models[("hebb", 42)]
hn_oji = models[("oji"), 42]

if_asynchronic = True

for i in range(patterns.shape[0]):
    hn_hebb.change_sync_mode(if_asynchronic)
    result = hn_hebb.recall(patterns[i], max_iter, True)
    if_correct_recalled = np.array_equal(patterns[i], result["Output states"])

    print("Model with Hebb method:")
    print(f"Correctly recalled: {if_correct_recalled}")
    print(f"Result: {result["Result"]}. Final energy value: {result["Energy"]}")
    print(f"Iterations: {result['Last iteration']}")
    hn.plot_history(result["History"], plot_dim)

for i in range(patterns.shape[0]):
    hn_oji.change_sync_mode(if_asynchronic)
    result = hn_oji.recall(patterns[i], max_iter, True)
    if_correct_recalled = np.array_equal(patterns[i], result["Output states"])

    print("Model with Oji method:")
    print(f"Correctly recalled: {if_correct_recalled}")
    print(f"Result: {result["Result"]}. Final energy value: {result["Energy"]}")
    print(f"Iterations: {result['Last iteration']}")
    hn.plot_history(result["History"], plot_dim)

In [ ]:
def make_noisy_pattern(pattern, rng, noise_level):
    noisy_pattern = pattern.copy()
    n_flips = round(noise_level * noisy_pattern.size)

    indices = rng.choice(noisy_pattern.size, size=n_flips, replace=False)

    noisy_pattern[indices] *= -1
    return noisy_pattern

In [ ]:
recall_results = []

for (method, seed), hn in models.items():
    for mode in modes:
        hn.change_sync_mode(mode == "async")

        for i in range(n_patterns):
            for j in range(noise_levels.shape[0]):
                for k in range(noise_per_pattern):
                    noise_rng = np.random.default_rng([noise_seed, i, j, k])
                    noisy_pattern = make_noisy_pattern(patterns[i], noise_rng, noise_levels[j])

                    hn.rng = np.random.default_rng([recall_seed, i, j, k])
                    result = hn.recall(noisy_pattern, max_iter=max_iter)

                    output = result["Output states"]
                    stable = result["Result"] == "stable"

                    correct = (stable and np.array_equal(patterns[i], output))
                    recall_results.append({
                        "Method": method,
                        "Mode": mode,
                        "Train seed": seed,
                        "Pattern": i,
                        "Noise level (%)": round(
                            noise_levels[j] * 100, 2
                        ),
                        "Noisy pattern number": k,
                        "Correct recall": correct,
                        "Stable": stable,
                        "Iterations": result["Last iteration"],
                        "Result": result["Result"],
                        "Energy": result["Energy"]
                    })

df_recall = pd.DataFrame(recall_results)

display(df_recall.head())
print(f"Total recall tests: {len(df_recall)}")

In [ ]:
df_seed_summary = (
    df_recall.groupby(["Method", "Mode", "Train seed", "Noise level (%)"],as_index=False).agg(
        Correct_recall=("Correct recall", "mean"),
        Stable_states=("Stable", "mean"),
        Mean_iterations=("Iterations", "mean"),
    )
)

df_seed_summary["Correct recall (%)"] = (df_seed_summary["Correct_recall"] * 100)
df_seed_summary["Stable states (%)"] = (df_seed_summary["Stable_states"] * 100)

display(df_seed_summary.head())

In [ ]:
df_summary = (
    df_seed_summary.groupby(["Method", "Mode", "Noise level (%)"], as_index=False).agg(
        Recall_mean=("Correct recall (%)", "mean"),
        Recall_std=("Correct recall (%)", "std"),
        Stability_mean=("Stable states (%)", "mean"),
        Stability_std=("Stable states (%)", "std"),
        Iterations_mean=("Mean_iterations", "mean"),
    )
)

display(df_summary)

## Generowanie wykresów

In [ ]:
plt.figure(figsize=(10, 5))

for (method, mode), group in df_summary.groupby(
    ["Method", "Mode"]
):
    x = group["Noise level (%)"].to_numpy()
    y = group["Recall_mean"].to_numpy()

    plt.plot(
        x, y,
        marker="o",
        label=f"{method.upper()} - {mode}"
    )

    if method == "oji":
        std = group["Recall_std"].to_numpy()

        plt.fill_between(
            x,
            np.maximum(y - std, 0),
            np.minimum(y + std, 100),
            alpha=0.15
        )

plt.xlabel("Noise level (%)")
plt.ylabel("Correct recall (%)")
plt.title(f"Recall accuracy - {data_name}")

plt.ylim(-5, 105)
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()

plt.savefig(
    output_dir / "rec_acc.png",
    bbox_inches="tight"
)

plt.show()

In [ ]:

plt.figure(figsize=(10, 5))

for (method, mode), group in df_summary.groupby(
    ["Method", "Mode"]
):
    plt.plot(
        group["Noise level (%)"],
        group["Stability_mean"],
        marker="o",
        label=f"{method.upper()} - {mode}"
    )

plt.xlabel("Noise level (%)")
plt.ylabel("Stable states (%)")
plt.title(f"Stabilization analysis - {data_name}")

plt.ylim(-5, 105)
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()

plt.savefig(
    output_dir / "stab_analysis.png",
    bbox_inches="tight"
)

plt.show()


In [ ]:
for mode in modes:
    for method in methods:
        df_selected = df_recall[(df_recall["Method"] == method) & (df_recall["Mode"] == mode)]

        df_patterns = (df_selected.groupby(["Pattern", "Noise level (%)"],as_index=False).agg(
                Correct_recall=("Correct recall", "mean")
            )
        )

        df_patterns["Correct recall (%)"] = (
            df_patterns["Correct_recall"] * 100
        )

        plt.figure(figsize=(10, 5))

        for pattern_id, group in df_patterns.groupby("Pattern"):
            plt.plot(
                group["Noise level (%)"],
                group["Correct recall (%)"],
                marker="o",
                label=f"Pattern {pattern_id}"
            )
            if int(pattern_id) > 6:
                break

        plt.xlabel("Noise level (%)")
        plt.ylabel("Correct recall (%)")
        plt.title(
            f"Individual patterns - {method} "
            f"({mode})"
        )

        plt.ylim(-5, 105)
        plt.grid(True, alpha=0.3)
        plt.legend()
        plt.tight_layout()

        plt.savefig(
            output_dir / f"individual_patterns_{method}_{mode}.png",
            bbox_inches="tight"
        )

        plt.show()
